# TerraVoce: verifica riproducibile del modello numerico

**Esito: calcoli riproducibili; previsione estiva non trasferibile ad autunno e inverno.**

Il modello qui esaminato è una regressione numerica dell'ossigeno a un'ora.
Il LLM locale non viene chiamato. La verifica Python usa un'altra implementazione
e un altro solver rispetto al TypeScript, ma le stesse misure USGS: non è una
revisione scientifica esterna o una misura indipendente sul campo.

Il quaderno usa soltanto la libreria standard Python e snapshot archiviati;
non richiede rete. Eseguirlo dalla radice del repository o da `docs/notebooks`.


In [1]:
from pathlib import Path
import importlib.util
import json

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/audit_willamette.py').exists())
spec = importlib.util.spec_from_file_location('willamette_audit', root / 'scripts/audit_willamette.py')
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
print('Fonti: USGS-14211720, ossigeno, temperatura e portata istantanea con segno')
print('Origine: snapshot locali con hash; nessuna chiamata di rete')


Fonti: USGS-14211720, ossigeno, temperatura e portata istantanea con segno
Origine: snapshot locali con hash; nessuna chiamata di rete


## 1. Ricostruzione del risultato estivo

Le serie grezze sono riabbinate per orario esatto. I coefficienti ridge vengono
ricalcolati in Python con Decimal a 50 cifre e Cholesky. Training, scelta,
calibrazione e verifica finale restano separati nel tempo.


In [2]:
original = audit.original_audit(root)
print(json.dumps({k: original[k] for k in ('passed', 'checkedCandidates', 'checkedHoldoutRows',
    'solver', 'numericTolerance', 'largestCoefficientOrMetricDifference', 'phaseCounts', 'selected')}, indent=2))


{
  "passed": true,
  "checkedCandidates": 14,
  "checkedHoldoutRows": 143,
  "solver": "Python stdlib Decimal Cholesky, 50 digits; no TypeScript imports",
  "numericTolerance": 1e-09,
  "largestCoefficientOrMetricDifference": 2.842170943040401e-14,
  "phaseCounts": {
    "training": 324,
    "selection": 95,
    "calibration": 95,
    "holdout": 143
  },
  "selected": "oxygen-clock-temperature:ridge-100"
}


## 2. Prova su stagioni diverse con modello congelato

Le finestre di ottobre 2025 e gennaio 2026 sono state fissate nel protocollo
pubblicato dal commit `16ffae5029cd2d426ce67e09f46f1b6c137b9a6c`, prima del download.
I coefficienti e la fascia di errore non vengono riaddestrati o ricalibrati.
Il riferimento semplice mantiene l'ultima misura. MAE significa errore assoluto
medio: valori più bassi sono migliori. Le finestre hanno durate e variabilità
diverse; la tabella non dimostra prestazioni per un anno intero.


In [3]:
seasonal = audit.frozen_validation(root)
report = json.loads((root / 'docs/evidence/willamette-forecast/forecast.json').read_bytes())
selected = next(c for c in report['candidates'] if c['name'] == report['selected'])
baseline = next(c for c in report['candidates'] if c['name'] == 'persistence')
print('Periodo          Coppie   MAE modello   MAE riferimento   Copertura')
print(f"Estate finale    143      {selected['holdout']['maeMgL']:.6f}      {baseline['holdout']['maeMgL']:.6f}          {report['interval']['holdoutCoverage']:.2%}")
for w in seasonal['windows']:
    print(f"{w['window']['id']:<16} {w['alignment']['acceptedPairs']:<8} {w['selectedModel']['maeMgL']:.6f}      {w['persistence']['maeMgL']:.6f}          {w['unchangedIntervalCoverage']:.2%}")
print('Nuovo fit:', seasonal['refitted'], '| Nuova calibrazione:', seasonal['recalibrated'])


Periodo          Coppie   MAE modello   MAE riferimento   Copertura
Estate finale    143      0.263681      0.267832          76.92%
autumn-2025      660      0.305686      0.101515          76.36%
winter-2026      645      1.278611      0.012403          0.00%
Nuovo fit: False | Nuova calibrazione: False


## 3. Condizioni diverse e dati mancanti

Un valore fuori dal minimo/massimo dell'addestramento è un segnale diagnostico,
non una probabilità di errore né una certificazione del dominio di validità.
Gli abbinamenti incompleti vengono esclusi; non si interpolano dati mancanti.
I dati sono archivi finalizzati: non dimostrano cosa fosse disponibile in tempo
reale alla data simulata. Nessuna finestra verifica eventi sotto 5 mg/L.


In [4]:
print('Intervalli di addestramento:')
for feature in seasonal['trainingFeatureRanges']:
    if feature['feature'] in ('oxygenNow', 'temperatureNow'):
        print(feature['feature'], feature['min'], '—', feature['max'])
for w in seasonal['windows']:
    print(w['window']['id'], '| coppie fuori intervallo:', w['featureExtrapolation']['fraction'],
          '| origini saltate:', w['alignment']['skipped'],
          '| bias mg/L:', round(w['selectedModel']['biasMgL'], 6),
          '| eventi sotto 5 mg/L:', w['observedTargetsBelowIllustrative5MgL'])


Intervalli di addestramento:
oxygenNow 7.3 — 9.0
temperatureNow 23.0 — 24.5
autumn-2025 | coppie fuori intervallo: 1.0 | origini saltate: 0 | bias mg/L: -0.296889 | eventi sotto 5 mg/L: 0
winter-2026 | coppie fuori intervallo: 1.0 | origini saltate: 13 | bias mg/L: -1.278611 | eventi sotto 5 mg/L: 0


## 4. Decisione e obiettivo originale

La regressione tende a riportare l'ossigeno verso il livello estivo appreso:
questo produce una sottostima invernale. Conserviamo il fallimento e il modello
originale, senza ritoccarli dopo aver visto i risultati.

TerraVoce mantiene coordinamento ambientale, evidenze pubbliche, comunità e
giovani nelle decisioni e finanziamenti per interventi verificabili.
Questo esperimento non autorizza pagamenti e non entra nell'Oracle operativo.
Non simula scarichi o dimostra causalità/benefici di interventi. Una nuova ricerca
richiede dati più rappresentativi e nuove finestre non usate nella scelta.


In [5]:
assert original['passed']
assert not seasonal['refitted'] and not seasonal['recalibrated']
assert not seasonal['operationalModelChanged']
assert all(w['selectedModel']['maeMgL'] > w['persistence']['maeMgL'] for w in seasonal['windows'])
print(seasonal['decision'])
print('Governance e finanziamenti: nessuna modifica operativa')


retain as an isolated research experiment; no operational promotion
Governance e finanziamenti: nessuna modifica operativa
